In [1]:
!pip install einops

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [3]:
import torch
import math
import numpy as np
import os

import torchvision
import torchvision.transforms as transforms
#import torch.optim as optim
#from torch.optim import Adam

import torch.nn as nn
from torch.autograd import Variable
import pathlib

from PIL import Image
from sklearn.model_selection import train_test_split

from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor
from torchvision import datasets, models, transforms

from torch.nn import Linear, ReLU, CrossEntropyLoss, Sequential, Conv2d, MaxPool2d, Module, Softmax, BatchNorm2d, Dropout
from torch.optim import Adam, SGD,RMSprop
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader

from einops import rearrange 
from torch import nn, einsum
#from einops.layers.torch import Rearrange, Reduce
#from timm.models.layers import trunc_normal_, DropPath

In [4]:
device =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [5]:
trg_root_data = "/content/drive/MyDrive/Datasets/Industrial_Surveillance_Dataset"
class_labels = {0:"Violent", 1:"Non-Violent"}
frame_path = "/frames/"
maxtoAdd = 40
nb_classes = 2
#nb_epoch = 10

In [6]:
def load_data(trg_root_data, maxtoAdd):
  image_path = []
  classes = []

  for i in class_labels:
    act_labels = str(i) + "." + class_labels[i]
    classes.append(act_labels)
    class_frame_folder = trg_root_data + frame_path + act_labels
    #print(class_frame_folder)
    sub_folder = [sub_folder for sub_folder in os.listdir(class_frame_folder)]
    sub_folder.sort()
    for j in sub_folder: 
      subset_folder = class_frame_folder + "/" + str(j)
      #print(subset_folder)
      file_list = [file_list for file_list in os.listdir(subset_folder)]
      file_list.sort()
      folder_len = len(file_list)

      #print(f"In folder {main_folder} length is :  {folder_len}")

      for n in range(0, maxtoAdd):
          file_path = subset_folder + "/" + file_list[n]
          #print(file_path)
          #break
          image_path.append(file_path)
  return image_path, classes

In [7]:
print("loading data")
image_data, classes_data = load_data(trg_root_data, maxtoAdd)

loading data


In [8]:
print(len(classes_data))
print(len(image_data))

2
12000


In [9]:
idx_to_class = {i:j for i, j in enumerate(classes_data)}
class_to_idx = {value:key for key,value in idx_to_class.items()}

In [10]:
img_path = image_data[0]
print(img_path)

/content/drive/MyDrive/Datasets/Industrial_Surveillance_Dataset/frames/0.Violent/Indus-violent-001/1.jpg


In [11]:
label = img_path.split('/')[7]
print(label)
labels = class_to_idx[label]
print(labels)

0.Violent
0


In [12]:
transform_image = transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),
                                transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))       
                                ]) 

#transform_label = transforms.Compose([transforms.ToTensor()])

In [13]:
import cv2
class IndustrialDataset(Dataset):
    def __init__(self, image_paths, transform=transform_image):
        self.image_paths = image_paths
        self.transform = transform_image
        #self.label_transform = transform_label

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_filepath = self.image_paths[idx]
        image = Image.open(image_filepath)
        image_tensor = self.transform(image)

        label = image_filepath.split('/')[7]
        label = class_to_idx[label]
        return image_tensor, label

In [14]:
dataset_load = IndustrialDataset(image_data,transform_image)

In [15]:
train_size = int(0.8 * len(dataset_load))
test_size = len(dataset_load) - train_size

train_set, test_set = torch.utils.data.random_split(dataset_load, [train_size, test_size])


In [16]:
print("loading done")

train_dl = DataLoader(train_set, batch_size = 20, shuffle= True)
test_dl = DataLoader(test_set, batch_size = 20, shuffle= True)

loading done


In [17]:
print(type(train_dl))

<class 'torch.utils.data.dataloader.DataLoader'>


In [18]:
print(len(train_dl))
print(train_dl)

480


In [19]:
print("Swin_tarnsformer....")

class CyclicShift(nn.Module):
    def __init__(self, displacement):
        super().__init__()
        self.displacement = displacement

    def forward(self, x):
        return torch.roll(x, shifts=(self.displacement, self.displacement), dims=(1, 2))


class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x


class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x):
        return self.net(x)


def create_mask(window_size, displacement, upper_lower, left_right):
    mask = torch.zeros(window_size ** 2, window_size ** 2)

    if upper_lower:
        mask[-displacement * window_size:, :-displacement * window_size] = float('-inf')
        mask[:-displacement * window_size, -displacement * window_size:] = float('-inf')

    if left_right:
        mask = rearrange(mask, '(h1 w1) (h2 w2) -> h1 w1 h2 w2', h1=window_size, h2=window_size)
        mask[:, -displacement:, :, :-displacement] = float('-inf')
        mask[:, :-displacement, :, -displacement:] = float('-inf')
        mask = rearrange(mask, 'h1 w1 h2 w2 -> (h1 w1) (h2 w2)')

    return mask


def get_relative_distances(window_size):
    indices = torch.tensor(np.array([[x, y] for x in range(window_size) for y in range(window_size)]))
    distances = indices[None, :, :] - indices[:, None, :]
    return distances


class WindowAttention(nn.Module):
    def __init__(self, dim, heads, head_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        inner_dim = head_dim * heads

        self.heads = heads
        self.scale = head_dim ** -0.5
        self.window_size = window_size
        self.relative_pos_embedding = relative_pos_embedding
        self.shifted = shifted

        if self.shifted:
            displacement = window_size // 2
            self.cyclic_shift = CyclicShift(-displacement)
            self.cyclic_back_shift = CyclicShift(displacement)
            self.upper_lower_mask = nn.Parameter(create_mask(window_size=window_size, displacement=displacement,
                                                             upper_lower=True, left_right=False), requires_grad=False)
            self.left_right_mask = nn.Parameter(create_mask(window_size=window_size, displacement=displacement,
                                                            upper_lower=False, left_right=True), requires_grad=False)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)

        if self.relative_pos_embedding:
            self.relative_indices = get_relative_distances(window_size) + window_size - 1
            self.pos_embedding = nn.Parameter(torch.randn(2 * window_size - 1, 2 * window_size - 1))
        else:
            self.pos_embedding = nn.Parameter(torch.randn(window_size ** 2, window_size ** 2))

        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        if self.shifted:
            x = self.cyclic_shift(x)

        b, n_h, n_w, _, h = *x.shape, self.heads

        qkv = self.to_qkv(x).chunk(3, dim=-1)
        nw_h = n_h // self.window_size
        nw_w = n_w // self.window_size

        q, k, v = map(
            lambda t: rearrange(t, 'b (nw_h w_h) (nw_w w_w) (h d) -> b h (nw_h nw_w) (w_h w_w) d',
                                h=h, w_h=self.window_size, w_w=self.window_size), qkv)

        dots = einsum('b h w i d, b h w j d -> b h w i j', q, k) * self.scale

        if self.relative_pos_embedding:
            dots += self.pos_embedding[self.relative_indices[:, :, 0], self.relative_indices[:, :, 1]]
        else:
            dots += self.pos_embedding

        if self.shifted:
            dots[:, :, -nw_w:] += self.upper_lower_mask
            dots[:, :, nw_w - 1::nw_w] += self.left_right_mask

        attn = dots.softmax(dim=-1)

        out = einsum('b h w i j, b h w j d -> b h w i d', attn, v)
        out = rearrange(out, 'b h (nw_h nw_w) (w_h w_w) d -> b (nw_h w_h) (nw_w w_w) (h d)',
                        h=h, w_h=self.window_size, w_w=self.window_size, nw_h=nw_h, nw_w=nw_w)
        out = self.to_out(out)

        if self.shifted:
            out = self.cyclic_back_shift(out)
        return out


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, head_dim, mlp_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        self.attention_block = Residual(PreNorm(dim, WindowAttention(dim=dim,
                                                                     heads=heads,
                                                                     head_dim=head_dim,
                                                                     shifted=shifted,
                                                                     window_size=window_size,
                                                                     relative_pos_embedding=relative_pos_embedding)))
        self.mlp_block = Residual(PreNorm(dim, FeedForward(dim=dim, hidden_dim=mlp_dim)))

    def forward(self, x):
        x = self.attention_block(x)
        x = self.mlp_block(x)
        return x

class BasicConv(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1, groups=1, relu=True, bn=True, bias=False):
        super(BasicConv, self).__init__()
        self.out_channels = out_planes
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(out_planes,eps=1e-5, momentum=0.01, affine=True) if bn else None
        self.relu = nn.ReLU() if relu else None

    def forward(self, x):
        x = self.conv(x)
        if self.bn is not None:
            x = self.bn(x)
        if self.relu is not None:
            x = self.relu(x)
        return x

class ChannelPool(nn.Module):
    def forward(self, x):
        return torch.cat( (torch.max(x,1)[0].unsqueeze(1), torch.mean(x,1).unsqueeze(1)), dim=1 )

class SpatialGate(nn.Module):
    def __init__(self):
        super(SpatialGate, self).__init__()
        kernel_size = 7
        self.compress = ChannelPool()
        self.spatial = BasicConv(2, 1, kernel_size, stride=1, padding=(kernel_size-1) // 2, relu=False)
    def forward(self, x):
        x_compress = self.compress(x)
        x_out = self.spatial(x_compress)
        scale = torch.sigmoid(x_out) # broadcasting
        return x * scale

class PatchMerging_conv(nn.Module):
    def __init__(self, in_channels, out_channels, downscaling_factor):
        super().__init__()
        self.downscaling_factor = downscaling_factor
        
        #self.patch_merge = nn.Conv2d(in_channels,out_channels,kernel_size=downscaling_factor, stride=downscaling_factor, padding=0)
        self.patch_merge = nn.Unfold(kernel_size=downscaling_factor, stride=downscaling_factor, padding=0)
        self.linear = nn.Linear(in_channels * downscaling_factor ** 2, out_channels)

    def forward(self, x):
        b, c, h, w = x.shape
        new_h, new_w = h // self.downscaling_factor, w // self.downscaling_factor

        #x = self.patch_merge(x).permute(0, 2, 3, 1)
        x = self.patch_merge(x).view(b, -1, new_h, new_w).permute(0, 2, 3, 1)
        x = self.linear(x)
        return x

class StageModule(nn.Module):
    def __init__(self, in_channels, hidden_dimension, layers, downscaling_factor, num_heads, head_dim, window_size,
                 relative_pos_embedding):
        super().__init__()
        assert layers % 2 == 0, 'Stage layers need to be divisible by 2 for regular and shifted block.'


        self.patch_partition = PatchMerging_conv(in_channels=in_channels, out_channels=hidden_dimension,
                                            downscaling_factor=downscaling_factor)
        self.layers = nn.ModuleList([])
        for _ in range(layers // 2):
            self.layers.append(nn.ModuleList([
                SwinBlock(dim=hidden_dimension, heads=num_heads, head_dim=head_dim, mlp_dim=hidden_dimension * 4,
                          shifted=False, window_size=window_size, relative_pos_embedding=relative_pos_embedding),
                SwinBlock(dim=hidden_dimension, heads=num_heads, head_dim=head_dim, mlp_dim=hidden_dimension * 4,
                          shifted=True, window_size=window_size, relative_pos_embedding=relative_pos_embedding),
            ]))

    def forward(self, x):
       
        x = self.patch_partition(x)
        for regular_block, shifted_block in self.layers:
            x = regular_block(x)
            x = shifted_block(x)
        return x.permute(0, 3, 1, 2)


class SwinTransformer(nn.Module):
    def __init__(self, *, hidden_dim, layers, heads, channels=3, num_classes=2, head_dim=32, window_size=7,
                 downscaling_factors=(4, 2, 2, 2), relative_pos_embedding=True):
        super().__init__()

        self.sap_att = SpatialGate()
        self.stage1 = StageModule(in_channels=channels, hidden_dimension=hidden_dim, layers=layers[0],
                                  downscaling_factor=downscaling_factors[0], num_heads=heads[0], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage2 = StageModule(in_channels=hidden_dim, hidden_dimension=hidden_dim * 2, layers=layers[1],
                                  downscaling_factor=downscaling_factors[1], num_heads=heads[1], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage3 = StageModule(in_channels=hidden_dim * 2, hidden_dimension=hidden_dim * 4, layers=layers[2],
                                  downscaling_factor=downscaling_factors[2], num_heads=heads[2], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        self.stage4 = StageModule(in_channels=hidden_dim * 4, hidden_dimension=hidden_dim * 8, layers=layers[3],
                                  downscaling_factor=downscaling_factors[3], num_heads=heads[3], head_dim=head_dim,
                                  window_size=window_size, relative_pos_embedding=relative_pos_embedding)
        
        #self.norm = nn.LayerNorm(hidden_dim * 8)
        #self.per = ops.Permute([0, 3, 1, 2]) # B H W C -> B C H W
        self.pool = nn.AdaptiveAvgPool2d(3)
        self.flat = nn.Flatten(1)
        self.fc1 = nn.Linear(9216, 1024)
        self.relu1 = nn.LeakyReLU(0.1)
        self.bn1 = nn.BatchNorm1d(1024)
        self.fc2 = nn.Linear(1024, 256)
        self.relu2 = nn.LeakyReLU(0.1)
        self.drop = nn.Dropout(0.5)
        self.fc3 = nn.Linear(256, 64)
        self.relu3 = nn.LeakyReLU(0.1)
        self.fc4 = nn.Linear(64, 16)
        self.relu4 = nn.LeakyReLU(0.1)
        self.fc5 = nn.Linear(16, 2)
        self.relu5 = nn.Sigmoid()

    def forward(self, img):
        x = self.sap_att(img)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x.permute(0,3,1,2)
        x = self.pool(x)
        x = self.flat(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.bn1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.drop(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.fc4(x)
        x = self.relu4(x)
        x = self.fc5(x)
        x = self.relu5(x)
        return x


def swin_t(hidden_dim=96, layers=(2, 2, 6, 2), heads=(3, 6, 12, 24), **kwargs):
    return SwinTransformer(hidden_dim=hidden_dim, layers=layers, heads=heads, **kwargs)


def swin_s(hidden_dim=96, layers=(2, 2, 18, 2), heads=(3, 6, 12, 24), **kwargs):
    return SwinTransformer(hidden_dim=hidden_dim, layers=layers, heads=heads, **kwargs)


def swin_b(hidden_dim=128, layers=(2, 2, 18, 2), heads=(4, 8, 16, 32), **kwargs):
    return SwinTransformer(hidden_dim=hidden_dim, layers=layers, heads=heads, **kwargs)


def swin_l(hidden_dim=192, layers=(2, 2, 18, 2), heads=(6, 12, 24, 48), **kwargs):
    return SwinTransformer(hidden_dim=hidden_dim, layers=layers, heads=heads, **kwargs)

Swin_tarnsformer....


**SWIN_Transformer_Base**

In [20]:
from torch.optim.rmsprop import RMSprop
model = SwinTransformer(
    hidden_dim=128,
    layers=(2, 2, 18, 2),
    heads=(4,8,16,32),
    channels=3,
    num_classes=2,
    head_dim=32,
    window_size=7,
    downscaling_factors=(4, 2, 2, 2),
    relative_pos_embedding=True
).to (device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = SGD(model.parameters(),momentum = 0.9, lr = 0.001)
#nb_epoch = 30
print(model)

SwinTransformer(
  (sap_att): SpatialGate(
    (compress): ChannelPool()
    (spatial): BasicConv(
      (conv): Conv2d(2, 1, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
      (bn): BatchNorm2d(1, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
    )
  )
  (stage1): StageModule(
    (patch_partition): PatchMerging_conv(
      (patch_merge): Unfold(kernel_size=4, dilation=1, padding=0, stride=4)
      (linear): Linear(in_features=48, out_features=128, bias=True)
    )
    (layers): ModuleList(
      (0): ModuleList(
        (0): SwinBlock(
          (attention_block): Residual(
            (fn): PreNorm(
              (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (fn): WindowAttention(
                (to_qkv): Linear(in_features=128, out_features=384, bias=False)
                (to_out): Linear(in_features=128, out_features=128, bias=True)
              )
            )
          )
          (mlp_block): Residual(
      

In [21]:
# We use the pre-defined number of epochs to determine how many iterations to train the network on
nb_epoch = 20
trainAccuracy = 0
totalTrainAccuracy = 0
for epoch in range(nb_epoch):
	#Load in the data in batches using the train_loader object
    for i, (images, labels) in enumerate(train_dl):  
        # Move tensors to the configured device
        images = images.to(device)
        labels = labels.to(device)
    
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        # Backward and optimize
        
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        trainAccuracy = (predicted == labels).float().sum().item()
        trainAccuracy = trainAccuracy / labels.size(0)
        totalTrainAccuracy += trainAccuracy

    print('Epoch [{}/{}]  Train Loss: {:.4f}, Train Accuracy: {:.4f}'.format(epoch+1, nb_epoch, loss.item(), trainAccuracy))

Epoch [1/20]  Train Loss: 0.6782, Train Accuracy: 0.5000
Epoch [2/20]  Train Loss: 0.4313, Train Accuracy: 1.0000
Epoch [3/20]  Train Loss: 0.4051, Train Accuracy: 0.9000
Epoch [4/20]  Train Loss: 0.4336, Train Accuracy: 0.9000
Epoch [5/20]  Train Loss: 0.3502, Train Accuracy: 0.9500
Epoch [6/20]  Train Loss: 0.3197, Train Accuracy: 1.0000
Epoch [7/20]  Train Loss: 0.3176, Train Accuracy: 1.0000
Epoch [8/20]  Train Loss: 0.3360, Train Accuracy: 1.0000
Epoch [9/20]  Train Loss: 0.3422, Train Accuracy: 0.9500
Epoch [10/20]  Train Loss: 0.3149, Train Accuracy: 1.0000
Epoch [11/20]  Train Loss: 0.3146, Train Accuracy: 1.0000
Epoch [12/20]  Train Loss: 0.3134, Train Accuracy: 1.0000
Epoch [13/20]  Train Loss: 0.3135, Train Accuracy: 1.0000
Epoch [14/20]  Train Loss: 0.3136, Train Accuracy: 1.0000
Epoch [15/20]  Train Loss: 0.3637, Train Accuracy: 0.9500
Epoch [16/20]  Train Loss: 0.4364, Train Accuracy: 0.9000
Epoch [17/20]  Train Loss: 0.3134, Train Accuracy: 1.0000
Epoch [18/20]  Train Lo

In [22]:
from sklearn.metrics import precision_score
with torch.no_grad():
    model.eval()
    correct = 0
    total = 0
    for images, labels in test_dl:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).float().sum().item()

        print(f"Actual Value : {labels}  Predicted Value : {predicted}")

    # precision = precision_score(labels, predicted,average='binary')
    print("\n")
    print("Accuracy of the network on the {} test images: {} %".format(total, 100 * correct / total))
    # print(precision)

Actual Value : tensor([0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
       device='cuda:0')  Predicted Value : tensor([0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
       device='cuda:0')
Actual Value : tensor([0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1],
       device='cuda:0')  Predicted Value : tensor([0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1],
       device='cuda:0')
Actual Value : tensor([1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0],
       device='cuda:0')  Predicted Value : tensor([1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0],
       device='cuda:0')
Actual Value : tensor([0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0],
       device='cuda:0')  Predicted Value : tensor([0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0],
       device='cuda:0')
Actual Value : tensor([1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1],
       device='cuda:0')  Predicted V